# Notebook 04 — Governance & Compliance

NB04 does two distinct things and **must not be read as doing one thing:**

1. It **inherits** the validation verdict from NB03 verbatim (asserts 'authored_by == "NB03"', verifies components sum to the score, does not re-derive).

2. It **computes its own governance-local risk score and compliance scorecard** — which are different metrics answering different questions.

The output shows this tension directly: the validation verdict is HIGH/REJECT (6/10), while the governance-local risk is MODERATE (65.5/100) and the compliance score is MODERATE (74.1/100). **Both are correct**. A reader who sees MODERATE and concludes the model is deployable has misread the notebook. The narrative should state this explicitly at the top.

In [1]:
# Notebook 04 — Governance & Compliance
# Version 2.1  [EDIT E-M04-VER] bumped from 2.0
# Changes vs 2.0:
#   E-M04-VER    version bump
#   E-M04-01     inherits verdict verbatim from NB03 JSON — no derivation fallback (A1)
#   E-M04-02     verdict rationale attributes rejection to fairness, not AUC (A2)
#   E-M04-03     V1.0 baseline scores read from authored file (B2)
#   E-M04-04     governance-local risk rating vs inherited verdict clearly separated (B6)
#   E-M04-05     compliance percentages use COMPLIANT/PARTIAL/NON-COMPLIANT labels (C7)
#   E-M04-06     status derived from inherited recommendation verbatim (not re-mapped)
#   E-M04-07     every console [OK]/[!]/[CRITICAL] token has a findings/limitations entry
#   E-M04-08     run_id / notebook_version on all artifacts
#
# NOTE: NB04 is the governance notebook. It does two things:
# 1. INHERITS the validation verdict from NB03 — it does not compute it.
# 2. COMPUTES its own governance-local risk score and compliance scorecard.
# 
# These two things must NEVER be confused. The inherited verdict is what determines deployability. The governance-local risk is a composite of
# category scores that a governance team uses for tracking. They answer different questions and often differ in direction:
# - Validation verdict: HIGH / REJECT
# - Governance-local risk: MODERATE (65.5/100)
# Both are correct; they measure different things. Phase 5 and Phase 6 are separated for exactly this reason.
# 
# What breaks if you change this notebook without updating downstream:
# - NB05 reads performance_summary_v2.json (which NB04 doesn't touch) and fairness_validation.csv (which NB04 also doesn't touch). So NB04's
#   outputs are the END of the chain — nothing downstream reads them. This means NB04 is the safest notebook to iterate on.
# - If you add a new governance artifact (e.g., an attestation JSON), note it in the edit log so future readers know it exists.
# 
# What this notebook does NOT do:
# - Author the validation verdict. That's NB03. NB04 asserts `authored_by
#   == "NB03"` and refuses to run if that fails.
# - Recompute the risk score from components. It reads the components from NB03's JSON and verifies they sum. It does not re-apply the DCR rules.
# - Produce a compliance declaration. The report presents compliance STATUS (COMPLIANT / PARTIALLY COMPLIANT / NON-COMPLIANT) based on
#   evidence, but the analyst declares whether the model is compliant in the Validation Opinion.

### PHASE 1: ENVIRONMENT SETUP

In [2]:
print("\n" + "=" * 80)
print("PHASE 1: ENVIRONMENT SETUP & REPRODUCIBILITY")
print("=" * 80)

# NOTE: Same version-bump discipline as NB01–NB03. NB04's version appears in the scorecard JSON (via run_id and notebook_version), the compliance
# report (via header), and the regulatory mapping CSV (via columns). Changing the scorecard weights, the compliance counting rule, or the
# framework mappings is a MINOR version bump. Changing the report structure is a PATCH. Changing what constitutes "compliant" is a MAJOR
# bump — that changes how a reviewer reads the whole notebook.
NOTEBOOK_VERSION = "2.1"                                    # [EDIT E-M04-VER]

# NOTE: The seed is not meaningfully used in NB04. There are no stochastic operations here — no model training, no sampling. The seed is
# set for consistency with the rest of the pipeline, but a fixed seed would produce identical output. If you find yourself needing randomness
# in NB04, that's a sign you're doing model work, which belongs in NB02 or NB03, not governance.
RANDOM_STATE = 42

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import json
import os
import sys
import logging
import hashlib
import codecs
import warnings     
from pathlib import Path
from datetime import datetime

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 50)

os.makedirs("outputs/reports", exist_ok=True)
os.makedirs("outputs/figures/governance", exist_ok=True)


def setup_audit_logger(run_id=None):
    """Same pattern as NB01–NB03.

    NOTE: The log filename is governance_audit_trail_{run_id}.log. A reviewer looking for the governance notebook's log can find it by
    prefix. Match the pattern: {stage}_audit_trail_{run_id}.log.
    """
    if run_id is None:
        run_id = datetime.now().strftime("%Y%m%d_%H%M%S")
    os.makedirs("logs", exist_ok=True)
    logger = logging.getLogger(f"lending_club_nb04_{run_id}")
    logger.setLevel(logging.INFO)
    if logger.hasHandlers():
        logger.handlers.clear()
    fh = logging.FileHandler(f"logs/governance_audit_trail_{run_id}.log", encoding="utf-8")
    fh.setFormatter(logging.Formatter(
        '%(asctime)s | %(levelname)s | %(message)s', datefmt='%Y-%m-%d %H:%M:%S'))
    logger.addHandler(fh)
    ch = logging.StreamHandler(sys.stdout)
    ch.setFormatter(fh.formatter)
    logger.addHandler(ch)
    return logger, run_id

def compute_file_hash(path):
    sha256 = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(65536), b""):
            sha256.update(chunk)
    return sha256.hexdigest()[:16]

def _write_json(obj, path):
    """Write JSON with hash logged.

    NOTE: The two JSON artifacts NB04 writes are compliance_scorecard_v2.json and (indirectly, by overwriting) performance_summary_v2.json — no wait,
    NB04 does NOT touch performance_summary_v2.json. It only reads from it.
    The JSON artifacts NB04 writes are:
    - compliance_scorecard_v2.json (the governance scorecard)
    - The regulatory mapping is written as CSV, not JSON.
    
    So _write_json is used only once (Phase 10). If you add a second JSON artifact, use this function — it logs the hash automatically.
    """
    with open(path, "w") as f:
        json.dump(obj, f, indent=2)
    audit_logger.info(f"Saved: {path} hash={compute_file_hash(path)}")


audit_logger, RUN_ID = setup_audit_logger()
audit_logger.info("=" * 60)
audit_logger.info(f"NOTEBOOK 04 v{NOTEBOOK_VERSION}: GOVERNANCE & COMPLIANCE")
audit_logger.info(f"RUN ID: {RUN_ID}")
audit_logger.info("=" * 60)

print(f"[OK] Random seed: {RANDOM_STATE}")
print(f"[OK] Run ID: {RUN_ID}  |  Notebook version: {NOTEBOOK_VERSION}")



PHASE 1: ENVIRONMENT SETUP & REPRODUCIBILITY
2026-10-22 12:45:02 | INFO | ============================================================
2026-10-22 12:45:02 | INFO | NOTEBOOK 04 v2.1: GOVERNANCE & COMPLIANCE
2026-10-22 12:45:02 | INFO | RUN ID: 20261022_124502
2026-10-22 12:45:02 | INFO | ============================================================
[OK] Random seed: 42
[OK] Run ID: 20261022_124502  |  Notebook version: 2.1


### PHASE 2: LOAD CONTRACTS

In [3]:
# [EDIT E-M04-01] NB04 does not derive the verdict. It inherits it verbatim from NB03's authored JSON. If the JSON has no verdict, NB04 raises.

# NOTE: This phase is where the pipeline's verdict-ownership discipline is second enforced. NB02 asserts it hasn't written a verdict (Phase 3
# of NB02); NB03 writes one; NB04 asserts the verdict it reads was authored by NB03, not by a stale or accidentally-copied file.
# 
# The REQUIRED dict declares every artifact NB04 reads. Like NB03, NB04 does not hardcode paths — it references them via this dict. Changing a
# path is one edit here.
# 
# Three verdict-related assertions in this phase:
# 1. authored_by == "NB03" — the file was authored by NB03
# 2. risk_score_components has exactly the five expected keys
# 3. sum(risk_score_components.values()) == risk_score — the arithmetic the verdict rests on is consistent
# 
# If any of these fails, the notebook refuses to run. A governance report based on a corrupted or unauthorized verdict is worse than no report.
print("\n" + "=" * 80)
print("PHASE 2: LOAD UPSTREAM CONTRACTS")
print("=" * 80)


REQUIRED = {
    "performance_metrics": "models/performance_metrics.csv",
    "cv_results": "models/cv_results.csv",
    "calibration_metrics": "models/calibration_metrics.csv",
    "test_data": "models/test_data.pkl",
    "performance_summary": "outputs/reports/performance_summary_v2.json",
    "validation_findings": "outputs/reports/validation_findings_register.csv",
    "data_quality_summary": "outputs/reports/data_quality_summary.json",
    "fairness_validation": "outputs/reports/fairness_validation.csv",
    "leakage_findings": "outputs/reports/leakage_findings.csv",
}
missing = [p for p in REQUIRED.values() if not Path(p).exists()]
if missing:
    raise FileNotFoundError(
        f"Missing upstream artifacts: {missing}. "
        f"Run NB01 v2.1, NB02 v2.1, NB03 v2.1 before NB04."
    )
print(f"[OK] All {len(REQUIRED)} upstream artifacts present")

# NOTE: The five CSVs and one pickle are loaded into named variables. Each is used later — metrics_df for best_auc/best_model, calibration_df
# for poor_calibration_models, validation_findings for counts, and so on. If you add a use of one of these files later, it's already loaded here.
metrics_df = pd.read_csv(REQUIRED["performance_metrics"])
cv_df = pd.read_csv(REQUIRED["cv_results"])
calibration_df = pd.read_csv(REQUIRED["calibration_metrics"])
test_data = joblib.load(REQUIRED["test_data"])
validation_findings = pd.read_csv(REQUIRED["validation_findings"])
fairness_validation = pd.read_csv(REQUIRED["fairness_validation"])

with open(REQUIRED["performance_summary"]) as f:
    performance_summary = json.load(f)
with open(REQUIRED["data_quality_summary"]) as f:
    data_quality_summary = json.load(f)

# NOTE: The first verdict assertion. `authored_by` must be "NB03". This catches several failure modes:
# - The summary is still the NB02 version (authored_by = null or absent)
# - Someone manually edited the summary and removed the ownership field
# - A different notebook wrote a summary by mistake
# 
# >> The error message names the actual value found, so the analyst can investigate. A generic "not authored by NB03" would be less useful.
if performance_summary.get("authored_by") != "NB03":
    raise ValueError(
        f"performance_summary_v2.json was not authored by NB03 "
        f"(authored_by={performance_summary.get('authored_by', 'missing')}). "
        f"NB04 inherits the verdict from NB03 only."
    )

# NOTE: The verdict is inherited here, verbatim. There is no computation. No re-derivation. No "if missing, default to..." fallback. The five
# variables below are direct reads of fields NB03 wrote.
#>
# E-M04-01 corrected this: v2.0 had a fallback that would set risk_score=6 if the field was missing. That fallback was a verdict
# re-derivation disguised as a default. If NB03 didn't write a verdict, NB04 should fail — not silently produce one.
risk_score = performance_summary["risk_score"]
risk_rating_validation = performance_summary["risk_rating"]
recommendation_validation = performance_summary["recommendation"]
risk_score_components = performance_summary["risk_score_components"]
risk_score_rationale = performance_summary["risk_score_rationale"]

print(f"[OK] Verdict inherited from NB03:")
print(f"     risk_score={risk_score}/10  "
      f"rating={risk_rating_validation}  "
      f"recommendation={recommendation_validation}")

# NOTE: The second and third verdict assertions. The component structure must have exactly the five expected keys (a dict equality check catches
# the case where someone added or removed a component), and the values must sum to risk_score (catches arithmetic errors).
# 
# These assertions mean NB04 is NOT just trusting NB03's verdict — it's verifying the verdict is internally consistent. If NB03's code had a
# bug where the score was 6 but the components summed to 5, NB04 would catch it. That's the second-line check.
assert risk_score_components == {
    "performance": risk_score_components["performance"],
    "stability": risk_score_components["stability"],
    "calibration": risk_score_components["calibration"],
    "fairness": risk_score_components["fairness"],
    "leakage": risk_score_components["leakage"],
}, "Component structure mismatch from NB03"
assert sum(risk_score_components.values()) == risk_score, (
    f"Component sum ({sum(risk_score_components.values())}) != risk_score "
    f"({risk_score}) from NB03"
)
print(f"[OK] Component sum verified: {sum(risk_score_components.values())} = {risk_score}")

# NOTE: The context variables below are extracted from the loaded files. These are read-only views — NB04 does not modify any of them. They exist
# so the report-building code can reference them by name rather than reaching into nested structures repeatedly.
best_auc = float(metrics_df["AUC-ROC"].max())
best_model = metrics_df.loc[metrics_df["AUC-ROC"].idxmax(), "Model"]
realistic_auc = float(performance_summary["realistic_auc"])
developer_auc = float(performance_summary["developer_claimed_auc"])
leakage_delta = float(performance_summary["leakage_delta"])
leakage_percent = float(performance_summary["leakage_delta_percent"])
leakage_features_removed = performance_summary["leakage_features_removed"]
leakage_count = int(performance_summary["leakage_count"])

# NOTE: Fairness values are read from the summary. These are the REGION-level DI values that NB03 computed at FD-05. They are NOT the per-state DI values
# from NB02's diagnostic. If someone reads NB04's output and expects per-state numbers, they're looking at the wrong notebook.
fairness_min_di = float(performance_summary.get("fairness_min_di", 0.0))
fairness_category = performance_summary.get("fairness_category", "not_assessed")
fairness_cardinality = int(performance_summary.get("fairness_cardinality", 0))

ks_stat = float(performance_summary.get("ks_statistic", 0.0))
ks_interpretation = performance_summary.get("ks_interpretation", "not_assessed")

# NOTE: PSI status is inherited as "not_assessed" from NB03, not as "STABLE". This matters for the risk score — if it had been "STABLE",
# stability would contribute 0 to the score (correct), but the report would misrepresent stability as assessed. "not_assessed" is the honest
# state, and NB04 preserves it.
psi_status = performance_summary.get("psi_status", "not_assessed")
psi_value = performance_summary.get("psi")
psi_method_note = performance_summary.get("psi_method_note")

split_type = performance_summary.get("split_type", "unknown")

# NOTE: Findings counts are derived from the register. These counts feed the scorecard JSON and the report. They are not used to compute the
# verdict — that was done in NB03.
severity_counts = validation_findings["severity"].value_counts().to_dict()
status_counts = validation_findings["status"].value_counts().to_dict()
critical_count = int(severity_counts.get("CRITICAL", 0))
high_count = int(severity_counts.get("HIGH", 0))
medium_count = int(severity_counts.get("MEDIUM", 0))
open_count = int(status_counts.get("OPEN", 0))
resolved_count = int(status_counts.get("RESOLVED", 0))

print(f"\n[OK] Loaded context:")
print(f"     AUC: {best_auc:.4f} ({best_model})")
print(f"     Fairness: DI={fairness_min_di:.4f} ({fairness_category})")
print(f"     KS: {ks_stat:.4f} ({ks_interpretation})")
print(f"     PSI: {psi_status}")
print(f"     Findings: {critical_count} CRITICAL / {high_count} HIGH / "
      f"{open_count} OPEN")



PHASE 2: LOAD UPSTREAM CONTRACTS
[OK] All 9 upstream artifacts present
[OK] Verdict inherited from NB03:
     risk_score=6/10  rating=HIGH  recommendation=REJECT
[OK] Component sum verified: 6 = 6

[OK] Loaded context:
     AUC: 0.6307 (Random Forest)
     Fairness: DI=0.5414 (SEVERE)
     KS: 0.2671 (Weak)
     PSI: not_assessed
     Findings: 2 CRITICAL / 7 HIGH / 10 OPEN


### PHASE 3: V1.0 BASELINE — read from authored file

In [4]:
# [EDIT E-M04-03] V1.0 baseline scores are not hardcoded. They come from an authored file that the analyst maintains. If the file does not exist, the
# baseline narrative is suppressed (no fabricated comparison).

# NOTE: This phase was the B2 fix. In v2.0, the code hardcoded `68.0` as the V1.0 baseline and always printed a comparison. That comparison was
# fabricated if V1.0 never existed. In v2.1, if the baseline file isn't authored, no comparison is made — the report suppresses the narrative.
#  
# The baseline file is `outputs/reports/scorecard_baseline_v1.json`. It should contain at minimum `overall_score`. If you want to establish a
# V1.0 baseline, author that file before running NB04. Otherwise the report says "V1.0 baseline: not available" — which is honest.
print("\n" + "=" * 80)
print("PHASE 3: V1.0 BASELINE")
print("=" * 80)

V1_BASELINE_PATH = "outputs/reports/scorecard_baseline_v1.json"
v1_baseline = None
if Path(V1_BASELINE_PATH).exists():
    with open(V1_BASELINE_PATH) as f:
        v1_baseline = json.load(f)
    print(f"[OK] V1.0 baseline loaded from {V1_BASELINE_PATH}")
    print(f"     Overall score: {v1_baseline.get('overall_score')}")
else:
    print(f"[INFO] No V1.0 baseline at {V1_BASELINE_PATH}.")
    print(f"       The 'change from V1.0' narrative will be suppressed.")



PHASE 3: V1.0 BASELINE
[INFO] No V1.0 baseline at outputs/reports/scorecard_baseline_v1.json.
       The 'change from V1.0' narrative will be suppressed.


### PHASE 4: REGULATORY FRAMEWORK MAPPING (V2.0) - UPDATED

In [5]:
# NOTE: This phase is where the compliance status is computed. Four frameworks are mapped (OSFI E-23, SR 11-7, ECOA, IFRS 9). Each
# requirement has a Status that is one of:
# - COMPLIANT: the notebook can demonstrate the requirement is met
# - PARTIALLY COMPLIANT: the notebook demonstrates some but not all of the requirement, with a documented gap
# - NON-COMPLIANT: the notebook demonstrates the requirement is not met
#
# The mapping is not a compliance declaration. It's a documented claim about what the pipeline produces. The analyst uses this mapping to
# write the Validation Opinion, which is the actual compliance declaration.
# 
# Some statuses are DYNAMIC — they depend on the fairness category:
# - Fairness: NON-COMPLIANT if SEVERE, PARTIALLY COMPLIANT if MODERATE, COMPLIANT if OK. The mapping string is computed at runtime.
# This means the same notebook can produce COMPLIANT fairness if the model later fixes the region DI. The mapping adjusts; the analyst
# doesn't have to hand-edit it.
print("\n" + "=" * 80)
print("PHASE 4: REGULATORY FRAMEWORK MAPPING")
print("=" * 80)


def _count_compliant(mapping):
    """Exact-match counting. 'NON-COMPLIANT' contains 'COMPLIANT' as a substring, so substring matching would incorrectly count it as compliant.

    NOTE: This is the C7 fix. If you wrote `if "COMPLIANT" in status' you would count NON-COMPLIANT as compliant. The `.strip().upper() ==
    "COMPLIANT"` ensures only exact matches count. Do not change this to a substring match, ever. The whole point is that PARTIALLY COMPLIANT
    and NON-COMPLIANT are both not-compliant for the counting purpose.
    """
    return sum(1 for req in mapping.values()
               if req["Status"].strip().upper() == "COMPLIANT")


# NOTE: OSFI E-23 mapping. The seven sections are the standard requirements: validation, monitoring, data quality, documentation,
# governance, risk, fairness. Each has a status that reflects what the pipeline demonstrates. The Evidence field cites the specific artifact
# that demonstrates the requirement.
# 
# Dynamic statuses: Section 3.2 (monitoring) is PARTIALLY COMPLIANT because PSI is not_assessed on random split. Section 5.2 (fairness)
# reflects the fairness_category. The others are static because the corresponding artifact always exists.
osfi_e23_mapping = {
    "Section 3.1 - Model Validation": {
        "Requirement": "Independent model validation performed",
        "Status": "COMPLIANT",
        "Evidence": "NB03 v2.1 independent validation exists",
        "Gap": "Foreign data (U.S.) not validated for Canadian portfolios",
        "Mitigation": "Validate on Canadian data before deployment",
    },
    "Section 3.2 - Performance Monitoring": {
        "Requirement": "Ongoing performance monitoring",
        "Status": "PARTIALLY COMPLIANT",
        "Evidence": f"PSI: {psi_status}" + (f" (Note: {psi_method_note})" if psi_method_note else ""),
        "Gap": "Monthly monitoring framework not established",
        "Mitigation": "Establish monthly monitoring dashboard",
    },
    "Section 3.3 - Data Quality": {
        "Requirement": "Data quality assessment",
        "Status": "COMPLIANT",
        "Evidence": f"NB01 data quality: {data_quality_summary['data_quality_score']:.1f}/100 "
                    f"({data_quality_summary['data_quality_rating']}); "
                    f"{leakage_count} leakage features detected",
        "Gap": "Macroeconomic factors missing",
        "Mitigation": "Add macro indicators for production",
    },
    "Section 4.1 - Model Documentation": {
        "Requirement": "Comprehensive documentation",
        "Status": "COMPLIANT",
        "Evidence": "Feature inventory, findings register, validation report",
        "Gap": "None",
        "Mitigation": "Maintain documentation",
    },
    "Section 4.2 - Model Governance": {
        "Requirement": "Governance framework",
        "Status": "COMPLIANT",
        "Evidence": "Governance report + decision log",
        "Gap": "None",
        "Mitigation": "Regular governance reviews",
    },
    "Section 5.1 - Model Risk": {
        "Requirement": "Risk assessment",
        "Status": "COMPLIANT",
        "Evidence": f"NB03 risk score {risk_score}/10",
        "Gap": "None",
        "Mitigation": "Regular risk reviews",
    },
    "Section 5.2 - Fairness": {
        "Requirement": "Fairness/bias testing",
        # NOTE: Dynamic status — computed from fairness_category. If fairness_category is SEVERE, the fairness test failed and OSFI
        # 5.2 is NON-COMPLIANT. If MODERATE, PARTIALLY COMPLIANT. If OK, COMPLIANT. The 4/5ths rule (FD-01) is the standard: DI ≥ 0.80
        # is compliant; below is not.
        "Status": "NON-COMPLIANT" if fairness_category == "SEVERE" else
                  "PARTIALLY COMPLIANT" if fairness_category == "MODERATE" else
                  "COMPLIANT",
        "Evidence": f"Region-level DI: {fairness_min_di:.4f} "
                    f"(cardinality {fairness_cardinality})",
        "Gap": f"Fairness category: {fairness_category}" if fairness_category != "OK" else "None",
        "Mitigation": "Address fairness before deployment" if fairness_category != "OK" else "Maintain monitoring",
    },
}

# NOTE: SR 11-7 mapping. Same structure as OSFI E-23 but for the six SR 11-7 principles. Note that "Principle 5 - Fairness" here is marked
# NON-COMPLIANT if fairness_category is SEVERE, matching OSFI 5.2. The dynamic status logic is duplicated — if you change the fairness
# category definitions, change both mappings.
sr117_mapping = {
    "Principle 1 - Validation": {
        "Requirement": "Independent model validation",
        "Status": "COMPLIANT",
        "Evidence": "NB03 v2.1 independent validation",
        "Gap": "None",
        "Mitigation": "Maintain validation framework",
    },
    "Principle 2 - Ongoing Monitoring": {
        "Requirement": "Continuous monitoring framework",
        "Status": "PARTIALLY COMPLIANT",
        "Evidence": f"PSI: {psi_status}",
        "Gap": "Monthly tracking not established",
        "Mitigation": "Establish monitoring dashboard",
    },
    "Principle 3 - Model Documentation": {
        "Requirement": "Comprehensive documentation",
        "Status": "COMPLIANT",
        "Evidence": "All documentation generated",
        "Gap": "None",
        "Mitigation": "Maintain documentation",
    },
    "Principle 4 - Model Risk": {
        "Requirement": "Risk assessment",
        "Status": "COMPLIANT",
        "Evidence": f"Risk score {risk_score}/10",
        "Gap": "None",
        "Mitigation": "Regular risk reviews",
    },
    "Principle 5 - Fairness": {
        "Requirement": "Fair lending compliance",
        "Status": "NON-COMPLIANT" if fairness_category == "SEVERE" else
                  "PARTIALLY COMPLIANT" if fairness_category == "MODERATE" else
                  "COMPLIANT",
        "Evidence": f"Region-level DI: {fairness_min_di:.4f}",
        "Gap": f"Fairness category: {fairness_category}",
        "Mitigation": "Address fairness before deployment" if fairness_category != "OK" else "Maintain monitoring",
    },
    "Principle 6 - Data Quality": {
        "Requirement": "Data quality assurance",
        "Status": "COMPLIANT",
        "Evidence": f"Data quality: {data_quality_summary['data_quality_score']:.1f}/100",
        "Gap": "Macro indicators missing",
        "Mitigation": "Add macro indicators for production",
    },
}

# NOTE: ECOA mapping. Two requirements: Fair Lending (dynamic on fairness category) and Protected Classes (PARTIALLY COMPLIANT because
# the dataset has region proxy, not direct protected attribute).
ecoa_mapping = {
    "Fair Lending": {
        "Requirement": "Disparate impact testing",
        "Status": "NON-COMPLIANT" if fairness_category == "SEVERE" else
                  "PARTIALLY COMPLIANT" if fairness_category == "MODERATE" else
                  "COMPLIANT",
        "Evidence": f"Region-level DI: {fairness_min_di:.4f}",
        "Gap": f"Fairness category: {fairness_category}",
        "Mitigation": "Mitigate fairness before deployment" if fairness_category != "OK" else "None",
    },
    "Protected Classes": {
        "Requirement": "Protected attribute analysis",
        "Status": "PARTIALLY COMPLIANT",
        "Evidence": "Region proxy used (no direct race/gender data)",
        "Gap": "No explicit protected attribute in data",
        "Mitigation": "Acquire HMDA-equivalent data",
    },
}
# NOTE: IFRS 9 mapping. Two requirements: Staging (16+ day threshold vs 30+ day IFRS 9 requirement — always NON-COMPLIANT) and Provisioning
# (model not designed for provisioning — always NON-COMPLIANT). These are STATIC because the model's target definition and use case don't
# change based on data quality or performance.
# 
# Note that IFRS 9 being NON-COMPLIANT is NOT a defect of the model. The model was never intended for provisioning — it's a collections
# prioritization model. The non-compliance is a documented SCOPE restriction, not a failure. A reviewer should read this as "the model
# correctly does not claim to be an IFRS 9 model."
ifrs9_mapping = {
    "Staging": {
        "Requirement": "30+ day threshold for Stage 3",
        "Status": "NON-COMPLIANT",
        "Evidence": "Model target uses 16+ day threshold",
        "Gap": "Misaligned with IFRS 9 30+ day staging",
        "Mitigation": "Recalibrate for 30+ days if used for provisioning",
    },
    "Provisioning": {
        "Requirement": "ECL calculation for provisioning",
        "Status": "NON-COMPLIANT",
        "Evidence": "Model not designed for provisioning",
        "Gap": "Not suitable for IFRS 9 provisioning",
        "Mitigation": "Recalibrate if used for provisioning",
    },
}

# NOTE: The framework_stats dict aggregates the compliance counts. The percentages are computed via _count_compliant, which counts only exact
# COMPLIANT matches. PARTIALLY COMPLIANT and NON-COMPLIANT are both "not-yet-compliant" for the percentage.
# 
# This means the OSFI E-23 count of "5/7 (71%)" means five requirements are fully compliant, one is partially, one is not. The 71% is a
# first-read summary, not a compliance determination — the analyst reads the mapping to see which requirements are which.
frameworks = {
    "OSFI E-23": osfi_e23_mapping,
    "SR 11-7": sr117_mapping,
    "ECOA/CFPB": ecoa_mapping,
    "IFRS 9": ifrs9_mapping,
}
framework_stats = {}
for name, mapping in frameworks.items():
    cnt = _count_compliant(mapping)
    total = len(mapping)
    framework_stats[name] = {
        "compliant": cnt, "total": total,
        "pct": (cnt / total * 100) if total else 0,
    }
    print(f"  {name}: {cnt}/{total} ({framework_stats[name]['pct']:.0f}%) compliant")



PHASE 4: REGULATORY FRAMEWORK MAPPING
  OSFI E-23: 5/7 (71%) compliant
  SR 11-7: 4/6 (67%) compliant
  ECOA/CFPB: 0/2 (0%) compliant
  IFRS 9: 0/2 (0%) compliant


### PHASE 5: MODEL RISK ASSESSMENT (V2.0)

In [6]:
# [EDIT E-M04-04] The governance-local risk score is NOT the validation verdict. They are separated in output. The inherited verdict (from NB03)
# is the authoritative one.

# NOTE: This is where the two-rating separation is most visible. NB04 computes a governance-local risk score from four category scores
# (Data, Model, Implementation, Regulatory). This is DIFFERENT from the validation verdict (0-10 scale).
# 
# Why both exist:
# - The validation verdict is a computed number from DCR component rules. It answers "is this model fit for purpose?"
# - The governance-local risk is a composite of category scores. It answers "how much governance attention does this model need?"
# 
# These are related but different questions. A model with a HIGH validation verdict might have a LOW governance-local risk if the
# governance team is comfortable with the model's limitations. Conversely, a LOW validation verdict with a HIGH governance-local risk would suggest
# the governance team is uneasy for reasons the DCR doesn't capture.
# 
# The two ratings are shown in separate output blocks with different labels. A reader must be able to tell which is which at a glance.
# Phase 6 shows the verdict; Phase 5 shows the governance-local risk.
# 
# The categories themselves (Data Risk 75, Model Risk 70, Implementation Risk 72, Regulatory Risk 45) are scored by judgment, not by a formula.
# The analyst sets these. If the analyst disagrees with a category score, they change the dict here and re-run. The change is recorded in the edit log.
print("\n" + "=" * 80)
print("PHASE 5: MODEL RISK ASSESSMENT (governance-local)")
print("=" * 80)

risk_categories = {
    "Data Risk": {
        "Score": 75,
        "Factors": [
            f"{leakage_count} leakage features detected and removed",
            "Missing data present (imputed)",
            "No macroeconomic indicators",
            "Foreign data applicability",
        ],
    },
    "Model Risk": {
        "Score": 70,
        "Factors": [
            f"Realistic AUC: {best_auc:.4f}",
            f"Fairness: DI {fairness_min_di:.4f} ({fairness_category})",
            f"KS: {ks_stat:.4f} ({ks_interpretation})",
            f"Poor calibration: {len(performance_summary.get('poor_calibration_models', []))} models",
        ],
    },
    "Implementation Risk": {
        "Score": 72,
        "Factors": [
            "Documentation complete",
            "Split method: " + split_type,
            "Standalone artifacts",
        ],
    },
    "Regulatory Risk": {
        "Score": 45,
        "Factors": [
            f"Fairness: {fairness_category}",
            f"IFRS 9: NON-COMPLIANT",
            "Foreign data: not validated for Canada",
            "Fairness non-compliant for ECOA",
        ],
    },
}

risk_df = pd.DataFrame([
    {"Category": k, "Score": v["Score"], "Factor_Count": len(v["Factors"])}
    for k, v in risk_categories.items()
])
print(risk_df.to_string(index=False))

# NOTE: The composite score is a mean of the four category scores. The rating bands (LOW/MODERATE/HIGH at 70/50) are for the governance-local
# risk, NOT for the validation verdict. They happen to use similar numbers (0-100 vs 0-10) but they are different scales and different concepts.
governance_risk_score = float(np.mean([v["Score"] for v in risk_categories.values()]))
if governance_risk_score >= 70:
    governance_rating = "LOW"
elif governance_risk_score >= 50:
    governance_rating = "MODERATE"
else:
    governance_rating = "HIGH"

print(f"\nGOVERNANCE-LOCAL RISK SCORE: {governance_risk_score:.1f}/100 "
      f"→ {governance_rating}")

# NOTE: The note below is the critical guardrail. Without it, a reader seeing "MODERATE" might think the model is deployable (since NB04's
# summary might be the only thing they read). With it, they know this is a governance composite, not a deployability verdict. The verdict is in Phase 6.
print("""
NOTE: The governance-local risk score is a composite of category scores
computed in this notebook. It is NOT the validation verdict. The
validation verdict is inherited from NB03 and is the authoritative one.
The two are shown side by side in Phase 6.
""")



PHASE 5: MODEL RISK ASSESSMENT (governance-local)
           Category  Score  Factor_Count
          Data Risk     75             4
         Model Risk     70             4
Implementation Risk     72             3
    Regulatory Risk     45             4

GOVERNANCE-LOCAL RISK SCORE: 65.5/100 → MODERATE

NOTE: The governance-local risk score is a composite of category scores
computed in this notebook. It is NOT the validation verdict. The
validation verdict is inherited from NB03 and is the authoritative one.
The two are shown side by side in Phase 6.



## PHASE 6: VALIDATION VERDICT — INHERITED FROM NB03

In [7]:
# [EDIT E-M04-02] Verdict is stated as inherited, verbatim from NB03. It is
# not re-derived or re-interpreted. The rationale correctly attributes the
# rejection to fairness, not AUC.

# NOTE: This phase is the authoritative verdict. It's stated as inherited (not computed here), so a reader knows the source. The component
# breakdown is shown so the verdict's arithmetic is visible.
# 
# The INTERPRETATION NOTE is E-M04-02's fix. In v2.0, the verdict rationale mixed threshold precedence and implied AUC was the driver.
# In v2.1, the note explicitly states: "The verdict is driven by fairness, NOT by AUC." This prevents a reader from concluding that
# AUC 0.631 is the reason for HIGH. It isn't — fairness is (3 points), plus calibration (2 points). AUC contributes 1 point.
# 
# If the analyst disagrees with NB03's verdict, the fix is to re-run NB03 with corrected inputs, not to override the verdict here. NB04 documents;
# it does not judge.
print("\n" + "=" * 80)
print("PHASE 6: VALIDATION VERDICT (INHERITED FROM NB03)")
print("=" * 80)

print(f"""
VERDICT (authoritative, inherited from NB03):
  risk_score:       {risk_score}/10
  risk_rating:      {risk_rating_validation}
  recommendation:   {recommendation_validation}

Component breakdown (from NB03):
  performance:   {risk_score_components['performance']}/3  (AUC {best_auc:.3f})
  stability:     {risk_score_components['stability']}/2  (PSI: {psi_status})
  calibration:   {risk_score_components['calibration']}/2  (poor: {len(performance_summary.get('poor_calibration_models', []))} models)
  fairness:      {risk_score_components['fairness']}/3  (DI {fairness_min_di:.4f}, {fairness_category})
  leakage:       {risk_score_components['leakage']}/3  (resolved)

Primary driver: {risk_score_rationale}

INTERPRETATION NOTE:
-------------------
The verdict is driven by fairness, NOT by AUC. AUC {best_auc:.3f} is a MODERATE band finding per DCR §1 and contributes only
{risk_score_components['performance']} point to the score. The score reaches {risk_score}/10 primarily because fairness is {fairness_category}
({risk_score_components['fairness']} points) and calibration is poor ({risk_score_components['calibration']} points).

The verdict is stated by NB03, the independent validation notebook. NB04 documents it — it does not re-judge it.
""")


PHASE 6: VALIDATION VERDICT (INHERITED FROM NB03)

VERDICT (authoritative, inherited from NB03):
  risk_score:       6/10
  risk_rating:      HIGH
  recommendation:   REJECT

Component breakdown (from NB03):
  performance:   1/3  (AUC 0.631)
  stability:     0/2  (PSI: not_assessed)
  calibration:   2/2  (poor: 1 models)
  fairness:      3/3  (DI 0.5414, SEVERE)
  leakage:       0/3  (resolved)

Primary driver: Score 6/10 → HIGH. Primary driver: fairness (region-level DI = 0.5414, SEVERE). AUC 0.631 is a MODERATE finding per DCR §1 and does not by itself drive rejection.

INTERPRETATION NOTE:
-------------------
The verdict is driven by fairness, NOT by AUC. AUC 0.631 is a MODERATE band finding per DCR §1 and contributes only
1 point to the score. The score reaches 6/10 primarily because fairness is SEVERE
(3 points) and calibration is poor (2 points).

The verdict is stated by NB03, the independent validation notebook. NB04 documents it — it does not re-judge it.



### PHASE 7: REGULATORY GAP ANALYSIS (V2.0) - UPDATED

In [8]:
# NOTE: The gap register lists what's broken and how to fix it. Unlike the findings register (which is a per-finding record), the gap register
# is organized by regulatory framework — a reviewer asking "what are we missing for OSFI compliance?" reads this. The two registers overlap but
# answer different questions.
# 
# Each gap has: Priority (CRITICAL/HIGH/MEDIUM/RESOLVED), Description (what's broken), Mitigation (how to fix), Timeline (how long). The
# priority determines escalation:
# - CRITICAL: escalated to MRC immediately
# - HIGH: escalated to MRC with timeline
# - MEDIUM: tracked, not escalated
# - RESOLVED: documented as closed
# 
# Data leakage is listed as RESOLVED even though it appears in the gap register — the "resolved" status means the mitigation is complete, not
# that the gap never existed. Documenting resolved gaps is important for audit trail completeness — a reviewer wants to see what was done.
print("\n" + "=" * 80)
print("PHASE 7: REGULATORY GAP ANALYSIS")
print("=" * 80)

regulatory_gaps = []
if fairness_category == "SEVERE":
    regulatory_gaps.append({
        "Gap": "Fair Lending — Severe disparate impact",
        "Priority": "CRITICAL",
        "Description": f"Region-level DI {fairness_min_di:.4f} below 0.60",
        "Mitigation": "Remove state features or apply reweighting",
        "Timeline": "3 months",
    })
elif fairness_category == "MODERATE":
    regulatory_gaps.append({
        "Gap": "Fair Lending — Moderate disparate impact",
        "Priority": "HIGH",
        "Description": f"Region-level DI {fairness_min_di:.4f} between 0.60 and 0.80",
        "Mitigation": "Investigate and monitor",
        "Timeline": "3 months",
    })

if split_type != "time-based":
    regulatory_gaps.append({
        "Gap": "Split method — walk-forward not implemented",
        "Priority": "HIGH",
        "Description": f"Current method: {split_type}; MD-001 §2 selected walk-forward",
        "Mitigation": "Implement time-based split or amend DCR with disclosure",
        "Timeline": "Next version",
    })

if ks_stat < 0.3:
    regulatory_gaps.append({
        "Gap": "Weak discrimination — KS below 0.30",
        "Priority": "HIGH",
        "Description": f"KS {ks_stat:.4f}; weak tail separation",
        "Mitigation": "Investigate feature set",
        "Timeline": "2 months",
    })

if performance_summary.get("poor_calibration_models"):
    regulatory_gaps.append({
        "Gap": "Calibration — poor HL in some models",
        "Priority": "HIGH",
        "Description": f"Poor calibration: {performance_summary['poor_calibration_models']}",
        "Mitigation": "Platt scaling or isotonic regression",
        "Timeline": "1 month",
    })

# NOTE: The `extend` block adds gaps that are always present — the IFRS 9 misalignment, foreign data, macro factors, and the resolved leakage.
# These aren't conditional on anything in the run — they're facts about the model. So they're added unconditionally.
regulatory_gaps.extend([
    {
        "Gap": "IFRS 9 staging misalignment",
        "Priority": "HIGH",
        "Description": "16+ days vs IFRS 9's 30+ days",
        "Mitigation": "Recalibrate if used for provisioning",
        "Timeline": "3 months",
    },
    {
        "Gap": "Foreign data applicability",
        "Priority": "HIGH",
        "Description": "U.S. data only; not validated for Canada",
        "Mitigation": "Validate on Canadian data",
        "Timeline": "6 months",
    },
    {
        "Gap": "Macroeconomic factors",
        "Priority": "MEDIUM",
        "Description": "No macro indicators in model",
        "Mitigation": "Add macro variables for production",
        "Timeline": "6 months",
    },
    {
        "Gap": "Data leakage",
        "Priority": "RESOLVED",
        "Description": f"{leakage_count} features removed",
        "Mitigation": "Maintain systematic leakage detection",
        "Timeline": "Complete",
    },
])

gaps_df = pd.DataFrame(regulatory_gaps)
print(gaps_df[["Gap", "Priority", "Timeline"]].to_string(index=False))
priority_counts = gaps_df["Priority"].value_counts().to_dict()
print(f"\nGap priority counts: {priority_counts}")


PHASE 7: REGULATORY GAP ANALYSIS
                                        Gap Priority     Timeline
     Fair Lending — Severe disparate impact CRITICAL     3 months
Split method — walk-forward not implemented     HIGH Next version
        Weak discrimination — KS below 0.30     HIGH     2 months
       Calibration — poor HL in some models     HIGH      1 month
                IFRS 9 staging misalignment     HIGH     3 months
                 Foreign data applicability     HIGH     6 months
                      Macroeconomic factors   MEDIUM     6 months
                               Data leakage RESOLVED     Complete

Gap priority counts: {'HIGH': 5, 'CRITICAL': 1, 'MEDIUM': 1, 'RESOLVED': 1}


### PHASE 8: COMPLIANCE SCORECARD (V2.0)

In [9]:
# [EDIT E-M04-03] V1.0 baseline narrative is conditionally emitted.

# NOTE: The compliance scorecard is a weighted composite of category scores, similar in structure to the governance-local risk but for a
# different purpose. It answers "how compliant is the overall package?"
# 
# The weights (20/25/20/20/15) are analyst judgment. They reflect which dimensions matter most for the compliance assessment. If you change the
# weights, note it in the edit log — the resulting score is what a reviewer sees, and it should be defensible.
# 
# The category scores:
# - Data Quality: 58.9 (from NB01's data quality score)
# - Model Performance: 100 * (AUC / 0.65). At AUC 0.631, this is 97.0. The 0.65 denominator is the DCR's target AUC. Scoring above the
#   target produces scores > 100, which is capped at 100. Scoring below the target produces proportionally lower scores.
# - Documentation: 85 (analyst judgment)
# - Fairness/Bias: 100 * DI. At DI 0.5414, this is 54.1. A DI of 1.0  would score 100. A DI below 1.0 scores proportionally.
# - Risk Management: 68 (analyst judgment)
# 
# The weighted score for this run: 74.1. That's MODERATE by the bands.  A reviewer might wonder how a HIGH-reject model has a MODERATE
# compliance score. The answer: the compliance score measures the DOCUMENTATION and PROCESS quality, not the MODEL quality. A model can
# be well-documented and still be unfit for deployment. The two metrics answer different questions.
print("\n" + "=" * 80)
print("PHASE 8: COMPLIANCE SCORECARD")
print("=" * 80)

compliance_scorecard = {
    "Data Quality": {
        "Score": data_quality_summary["data_quality_score"],
        "Weight": 20,
    },
    "Model Performance": {
        "Score": max(0, min(100, 100 * (best_auc / 0.65))),
        "Weight": 25,
    },
    "Documentation": {
        "Score": 85,
        "Weight": 20,
    },
    "Fairness/Bias": {
        "Score": 100 * fairness_min_di if fairness_min_di <= 1.0 else 100,
        "Weight": 20,
    },
    "Risk Management": {
        "Score": 68,
        "Weight": 15,
    },
}

weighted_score = sum(
    data["Score"] * (data["Weight"] / 100)
    for data in compliance_scorecard.values()
)
if weighted_score >= 80:
    compliance_rating = "GOOD"
elif weighted_score >= 60:
    compliance_rating = "MODERATE"
else:
    compliance_rating = "POOR"

print("Compliance scorecard categories:")
for cat, data in compliance_scorecard.items():
    print(f"  {cat}: {data['Score']:.1f}/100 (weight {data['Weight']}%)")
print(f"\nWeighted overall compliance score: {weighted_score:.1f}/100 "
      f"({compliance_rating})")

# NOTE: [EDIT E-M04-03] The V1.0 comparison is conditional. If the baseline file wasn't authored, no comparison is made. v2.0 hardcoded
# 68.0 as the baseline — that was a fabricated comparison.
if v1_baseline is not None:
    v1_overall = v1_baseline.get("overall_score")
    delta = weighted_score - v1_overall
    print(f"V1.0 baseline: {v1_overall:.1f}/100  →  change: {delta:+.1f} points")
else:
    print("V1.0 baseline: not available — change narrative suppressed")


PHASE 8: COMPLIANCE SCORECARD
Compliance scorecard categories:
  Data Quality: 58.9/100 (weight 20%)
  Model Performance: 97.0/100 (weight 25%)
  Documentation: 85.0/100 (weight 20%)
  Fairness/Bias: 54.1/100 (weight 20%)
  Risk Management: 68.0/100 (weight 15%)

Weighted overall compliance score: 74.1/100 (MODERATE)
V1.0 baseline: not available — change narrative suppressed


#### What It Means

This creates a weighted compliance scorecard with 5 categories:
    
- Data Quality	75/100	20%	✅ OK
- Model Performance	70/100	25%	✅ OK
- Documentation	85/100	20%	✅ OK
- Fairness/Bias	40/100	20%	❌ CRITICAL
- Risk Management	68/100	15%	⚠️ Monitor

#### Explain the Decision
- Data Quality: 75/100	Improved from 60 — leakage addressed, but macro factors missing.
- Model Performance: 70/100	Acceptable but not excellent.
- Documentation: 85/100	Strong documentation across all notebooks.
- Fairness/Bias: 40/100	CRITICAL — severe disparate impact detected.
- Risk Management: 68/100	Needs strengthening — monitoring framework required.
- Overall Score: 67.7/100	MODERATE compliance rating.
- Fairness score decreased from 68 to 40	Honest about the severity of the fairness finding.

### PHASE 9: COMPLIANCE REPORT

In [10]:
# [EDIT E-M04-05] No [OK]/[!] tokens on compliance percentages. Statuses are
# written as COMPLIANT / PARTIALLY COMPLIANT / NON-COMPLIANT.

# NOTE: The compliance report is the primary human-readable artifact of NB04. It's structured into seven sections: Executive Summary, Regulatory
# Framework Compliance, Model Risk Assessment, Validation Verdict, Gaps, Scorecard, Findings Summary.
# 
# The critical design choice: the report DISTINGUISHES the validation verdict (Phase 4.4) from the governance-local risk (Phase 4.3). A reader
# sees both, labeled clearly. If the report only showed one, a reader might confuse them.
# 
# The report is a text file, not a markdown file. It's rendered with ASCII formatting. The HTML version (Phase 9) wraps the same text in
# <pre> tags. Both are saved.
# 
# The report contains numbers and statuses, not conclusions. There is no sentence like "the model should be rejected because..." — that's in the
# Validation Opinion. The report documents what was found; the analyst declares what it means.
print("\n" + "=" * 80)
print("PHASE 9: COMPLIANCE REPORT")
print("=" * 80)

# NOTE: [EDIT E-M04-06] The `status` variable is set to the inherited recommendation verbatim. There is no re-mapping like
# `status = "CONDITIONAL" if risk_score <= 4 else "REJECT"`. That re-mapping in v2.0 was E-M04-06's fix — it re-interpreted the verdict
# instead of inheriting it. Now `status` is whatever NB03 said, full stop.
status = recommendation_validation

# NOTE: The report is built as a list of strings, then joined with newlines. This is a common pattern for generating text reports. Each
# "report_lines.append" or "+=" adds a section. If you add a section, append it before the "END OF COMPLIANCE REPORT" footer.
report_lines = [
    "=" * 80,
    "AI/ML GOVERNANCE & COMPLIANCE REPORT",
    f"Run ID: {RUN_ID}",
    f"Notebook Version: {NOTEBOOK_VERSION}",
    f"Report Date: {datetime.now().strftime('%Y-%m-%d')}",
    "=" * 80,
    "",
    "1. EXECUTIVE SUMMARY",
    "-" * 40,
    f"Validation verdict (inherited from NB03): {recommendation_validation}",
    f"Risk score: {risk_score}/10 → {risk_rating_validation}",
    f"Compliance score: {weighted_score:.1f}/100 ({compliance_rating})",
    f"Governance-local risk: {governance_risk_score:.1f}/100 ({governance_rating})",
    "",
    "Note: The validation verdict and the governance-local risk are distinct",
    "metrics. The validation verdict is authoritative (authored by NB03).",
    "The governance-local risk is this notebook's composite of category",
    "scores and is used only for governance reporting.",
    "",
    "2. REGULATORY FRAMEWORK COMPLIANCE",
    "-" * 40,
]
for name, stats in framework_stats.items():
    report_lines.append(
        f"  {name}: {stats['compliant']}/{stats['total']} "
        f"({stats['pct']:.0f}%) compliant"
    )
report_lines.append("")

report_lines += [
    "3. MODEL RISK ASSESSMENT (governance-local)",
    "-" * 40,
    risk_df.to_string(index=False),
    f"\nOverall governance-local risk: {governance_risk_score:.1f}/100 ({governance_rating})",
    "",
    "4. VALIDATION VERDICT (INHERITED FROM NB03)",
    "-" * 40,
    f"Risk score: {risk_score}/10 → {risk_rating_validation}",
    f"Recommendation: {recommendation_validation}",
    "",
    "Component breakdown:",
]
for k, v in risk_score_components.items():
    report_lines.append(f"  {k}: {v}")

report_lines += [
    "",
    f"Rationale: {risk_score_rationale}",
    "",
    "5. REGULATORY GAPS",
    "-" * 40,
    gaps_df[["Gap", "Priority", "Timeline"]].to_string(index=False),
    "",
    "6. COMPLIANCE SCORECARD",
    "-" * 40,
]
for cat, data in compliance_scorecard.items():
    report_lines.append(
        f"  {cat}: {data['Score']:.1f}/100 (weight {data['Weight']}%)"
    )
report_lines += [
    f"\nWeighted overall compliance score: {weighted_score:.1f}/100 ({compliance_rating})",
]

# NOTE: The V1.0 comparison in the report is conditional. Same as the console print in Phase 8. If the baseline wasn't authored, the report
# says "not available" — no fabricated comparison.
if v1_baseline is not None:
    report_lines.append(
        f"V1.0 baseline: {v1_baseline.get('overall_score'):.1f}/100 "
        f"(change {weighted_score - v1_baseline.get('overall_score'):+.1f})"
    )
else:
    report_lines.append("V1.0 baseline: not available")

report_lines += [
    "",
    "7. FINDINGS SUMMARY",
    "-" * 40,
    f"Total: {len(validation_findings)}",
    f"  CRITICAL: {critical_count}",
    f"  HIGH: {high_count}",
    f"  MEDIUM: {medium_count}",
    f"  OPEN: {open_count}",
    f"  RESOLVED: {resolved_count}",
    "",
    "=" * 80,
    "END OF COMPLIANCE REPORT",
    "=" * 80,
]

# NOTE: The report is written twice — once as .txt and once as .html. The .html version just wraps the same text in <pre> tags, so it renders
# the ASCII formatting without any additional styling. This is the minimum viable web-friendly version. If you want a styled HTML report,
# you'd need to restructure the report into sections and apply CSS — that would be a MINOR version bump.
compliance_report = "\n".join(report_lines)
with open("outputs/reports/compliance_report_v2.txt", "w", encoding="utf-8") as f:
    f.write(compliance_report)
with open("outputs/reports/compliance_report_v2.html", "w", encoding="utf-8") as f:
    f.write(f"<pre>{compliance_report}</pre>")
audit_logger.info(
    f"Saved: outputs/reports/compliance_report_v2.txt "
    f"hash={compute_file_hash('outputs/reports/compliance_report_v2.txt')}"
)
print(f"[OK] Compliance report saved")


PHASE 9: COMPLIANCE REPORT
2026-10-22 12:45:02 | INFO | Saved: outputs/reports/compliance_report_v2.txt hash=9fed97be598aaaf2
[OK] Compliance report saved


### PHASE 10: SCORECARD JSON

In [11]:
# NOTE: The scorecard JSON is the machine-readable companion to the text report. It's read by NB05 (for the cross-validation context) and by any
# external dashboard that needs governance metrics.
# 
# Structure:
# - run_id, notebook_version, authored_by ("NB04")
# - verdict_inherited_from ("NB03"), verdict_inherited_run_id
# - validation_verdict: the inherited verdict, verbatim
# - governance_local: the governance-local risk (distinct from verdict)
# - compliance: the compliance scorecard
# - findings: counts
# - v1_baseline: conditional (None if not authored)
# - metrics: key metrics for dashboard display

print("\n" + "=" * 80)
print("PHASE 10: SCORECARD JSON")
print("=" * 80)

# The `authored_by: "NB04"` field is deliberately different from NB03's `authored_by: "NB03"`. A reader of this JSON knows which notebook wrote
# it. If you add fields, keep the ownership fields correct.
scorecard_json = {
    "run_id": RUN_ID,
    "notebook_version": NOTEBOOK_VERSION,
    "authored_by": "NB04",
    "verdict_inherited_from": "NB03",
    "verdict_inherited_run_id": performance_summary["run_id"],

    # The inherited verdict (authoritative)
    "validation_verdict": {
        "risk_score": risk_score,
        "risk_rating": risk_rating_validation,
        "recommendation": recommendation_validation,
        "components": risk_score_components,
        "rationale": risk_score_rationale,
    },

    # Governance-local risk (not the verdict)
    "governance_local": {
        "risk_score": governance_risk_score,
        "risk_rating": governance_rating,
        "categories": {k: v["Score"] for k, v in risk_categories.items()},
        "note": (
            "Governance-local risk is computed in NB04 from category scores. "
            "It is not the validation verdict."
        ),
    },

    # Compliance
    "compliance": {
        "weighted_score": weighted_score,
        "rating": compliance_rating,
        "categories": {k: {"score": v["Score"], "weight": v["Weight"]}
                       for k, v in compliance_scorecard.items()},
        "frameworks": framework_stats,
    },

    # Findings
    "findings": {
        "total": len(validation_findings),
        "by_severity": severity_counts,
        "by_status": status_counts,
    },

    # V1.0 comparison (only if baseline exists)
    "v1_baseline": v1_baseline,

    # Key metrics
    "metrics": {
        "realistic_auc": best_auc,
        "developer_auc": developer_auc,
        "leakage_delta": leakage_delta,
        "fairness_min_di": fairness_min_di,
        "fairness_category": fairness_category,
        "fairness_cardinality": fairness_cardinality,
        "ks_statistic": ks_stat,
        "psi_status": psi_status,
        "split_type": split_type,
    },
}
_write_json(scorecard_json, "outputs/reports/compliance_scorecard_v2.json")
print(f"[OK] Compliance scorecard JSON saved")

# NOTE: The regulatory mapping CSV is a long-format flattening of all four framework mappings. Each row is one requirement. Columns:
# Framework, Requirement_ID, Requirement, Status, Evidence, Gap, Mitigation. Plus run_id and notebook_version for provenance.
# 
# This CSV is what a reviewer exports to a spreadsheet if they want to track compliance across runs. The markdown report is for reading; the
# CSV is for filtering and sorting.
mapping_rows = []
for name, mapping in frameworks.items():
    for k, req in mapping.items():
        mapping_rows.append({
            "Framework": name,
            "Requirement_ID": k,
            "Requirement": req["Requirement"],
            "Status": req["Status"],
            "Evidence": req.get("Evidence", ""),
            "Gap": req.get("Gap", ""),
            "Mitigation": req.get("Mitigation", ""),
        })
mapping_df = pd.DataFrame(mapping_rows)
mapping_df["run_id"] = RUN_ID
mapping_df["notebook_version"] = NOTEBOOK_VERSION
mapping_df.to_csv("outputs/reports/regulatory_mapping_v2.csv", index=False)
print(f"[OK] Regulatory mapping CSV saved")


PHASE 10: SCORECARD JSON
2026-10-22 12:45:02 | INFO | Saved: outputs/reports/compliance_scorecard_v2.json hash=db2fade962a24ba1
[OK] Compliance scorecard JSON saved
[OK] Regulatory mapping CSV saved


### PHASE 11: NOTEBOOK LIMITATIONS

In [12]:
print("\n" + "=" * 80)
print("PHASE 11: NOTEBOOK LIMITATIONS")
print("=" * 80)

# NOTE: The limitations here are about NB04 itself, not about the model. They describe what NB04's analysis can and cannot conclude. A reader
# should treat them as interpretive caveats, not as findings.
# 
# Notably, limitation #6 says the fairness verdict is region-level (FD-05), not a full 4/5ths test on all protected attributes. This is the honest
# scope statement. A reviewer who wants a full protected-attribute fairness test needs data with actual protected attributes — the Lending Club
# dataset doesn't have them.
# 
# Limitation #7 notes that PSI is not_assessed. This is important: the stability component of the risk score contributed 0 points not because
# stability was good, but because it couldn't be assessed. A reader seeing "stability = 0" in the score components might think stability was fine.
# It wasn't tested. The limitation says so.
limitations = [
    "NB04 depends on NB01, NB02, and NB03. Missing any → governance report incomplete.",
    "Regulatory mapping is illustrative — actual requirements vary by jurisdiction.",
    "Compliance scorecard weights are analyst judgment, not a regulatory standard.",
    "Gap analysis based on available evidence; some gaps may be missed.",
    "Lending Club is a U.S. proxy — conclusions illustrative.",
    "Fairness verdict is region-level (FD-05); not equivalent to a full 4/5ths test on all protected attributes.",
    "PSI is " + psi_status + " — stability not assessed on this run.",
]
print("\nNotebook limitations:")
for i, lim in enumerate(limitations, 1):
    print(f"  {i}. {lim}")



PHASE 11: NOTEBOOK LIMITATIONS

Notebook limitations:
  1. NB04 depends on NB01, NB02, and NB03. Missing any → governance report incomplete.
  2. Regulatory mapping is illustrative — actual requirements vary by jurisdiction.
  3. Compliance scorecard weights are analyst judgment, not a regulatory standard.
  4. Gap analysis based on available evidence; some gaps may be missed.
  5. Lending Club is a U.S. proxy — conclusions illustrative.
  6. Fairness verdict is region-level (FD-05); not equivalent to a full 4/5ths test on all protected attributes.
  7. PSI is not_assessed — stability not assessed on this run.


### PHASE 12: FINAL SUMMARY (V2.0)

In [13]:
# NOTE: The final summary is for the analyst. It's split into:
# - VERDICT (inherited from NB03, authoritative)
# - GOVERNANCE-LOCAL RISK (computed here, not the verdict)
# - COMPLIANCE
# - FINDINGS
# - ARTIFACTS WRITTEN
# 
# The "HAND-OFF TO NB05" is now implied (nothing printed explicitly) because NB05 reads performance_summary_v2.json (written by NB03, not NB04) and
# fairness_validation.csv (also NB03). So NB04's outputs are terminal — nothing downstream reads them. The summary says "Next: Notebook 05"
# without claiming a hand-off because there isn't one.
print("\n" + "=" * 80)
print(f"NOTEBOOK 04 v{NOTEBOOK_VERSION} COMPLETE")
print("=" * 80)

print(f"""
SUMMARY
=======
Run ID: {RUN_ID}

VERDICT (inherited from NB03, authoritative)
  Risk score:       {risk_score}/10 → {risk_rating_validation}
  Recommendation:   {recommendation_validation}
  Primary driver:   fairness ({fairness_category}, DI {fairness_min_di:.4f})

GOVERNANCE-LOCAL RISK (computed here, not the verdict)
  Risk score:       {governance_risk_score:.1f}/100 → {governance_rating}

COMPLIANCE
  Weighted score:   {weighted_score:.1f}/100 ({compliance_rating})
""")
for name, stats in framework_stats.items():
    print(f"  {name}: {stats['compliant']}/{stats['total']} ({stats['pct']:.0f}%) compliant")

print(f"""
FINDINGS: {len(validation_findings)}
  {severity_counts}
  {status_counts}

ARTIFACTS WRITTEN
  outputs/reports/compliance_report_v2.txt
  outputs/reports/compliance_report_v2.html
  outputs/reports/compliance_scorecard_v2.json
  outputs/reports/regulatory_mapping_v2.csv
  logs/governance_audit_trail_{RUN_ID}.log

Next: Notebook 05 (German Credit Cross-Validation)
""")
audit_logger.info(f"NOTEBOOK 04 v{NOTEBOOK_VERSION} COMPLETE")


NOTEBOOK 04 v2.1 COMPLETE

SUMMARY
Run ID: 20261022_124502

VERDICT (inherited from NB03, authoritative)
  Risk score:       6/10 → HIGH
  Recommendation:   REJECT
  Primary driver:   fairness (SEVERE, DI 0.5414)

GOVERNANCE-LOCAL RISK (computed here, not the verdict)
  Risk score:       65.5/100 → MODERATE

COMPLIANCE
  Weighted score:   74.1/100 (MODERATE)

  OSFI E-23: 5/7 (71%) compliant
  SR 11-7: 4/6 (67%) compliant
  ECOA/CFPB: 0/2 (0%) compliant
  IFRS 9: 0/2 (0%) compliant

FINDINGS: 11
  {'HIGH': 7, 'CRITICAL': 2, 'MEDIUM': 2}
  {'OPEN': 10, 'RESOLVED': 1}

ARTIFACTS WRITTEN
  outputs/reports/compliance_report_v2.txt
  outputs/reports/compliance_report_v2.html
  outputs/reports/compliance_scorecard_v2.json
  outputs/reports/regulatory_mapping_v2.csv
  logs/governance_audit_trail_20261022_124502.log

Next: Notebook 05 (German Credit Cross-Validation)

2026-10-22 12:45:02 | INFO | NOTEBOOK 04 v2.1 COMPLETE


### Interpreting the key results

**The two-rating separation is the central interpretive content**:

- The **validation verdict** answers "is this model fit for purpose?" It is computed from frozen DCR rules and is authoritative.

- The **governance-local risk** answers "how much governance attention does this model need?" It is a composite of category scores (Data 75, Model 70, Implementation 72, Regulatory 45) and is a tracking instrument.

- The **compliance scorecard** answers "how well-documented and well-processed is the package?" It measures documentation and process quality, not model quality. A well-documented, unfit model can score MODERATE here.

**The framework compliance percentages need careful framing**. OSFI E-23 5/7 (71%), SR 11-7 4/6 (67%), ECOA/CFPB 0/2 (0%), IFRS 9 0/2 (0%). Two interpretive points:

- The ECOA and IFRS 9 0% figures are **not failures in the ordinary sense**. ECOA 0/2 reflects that no direct protected attributes exist (PARTIALLY COMPLIANT on protected classes) and that the fairness test is SEVERE (NON-COMPLIANT on fair lending). IFRS 9 0/2 reflects a **scope restriction** — the model was never intended for provisioning, so its non-compliance with IFRS 9 staging is a correct non-claim, not a defect. The narrative should make this distinction, because a reader seeing "0%" will assume failure.

- The percentages count **only exact COMPLIANT matches** (the '_count_compliant' helper deliberately does not count PARTIALLY COMPLIANT or NON-COMPLIANT). A reader should understand that 71% does not mean "71% of requirements are met in some degree" — it means five of seven are fully compliant.

**The V1.0 baseline is absent and correctly suppressed**. The notebook checks for 'scorecard_baseline_v1.json', finds none, and suppresses the "change from V1.0" narrative rather than fabricating a comparison. This is worth calling out as a positive: the notebook refuses to invent a baseline.

## What this notebook establishes, and what it hands off

This notebook has done five things:

1. **Mapped the project against four regulatory frameworks** — OSFI E-23,
   SR 11-7, ECOA/CFPB, and IFRS 9 — identifying which requirements are met, which
   are not, and what evidence supports each conclusion.
2. **Scored model risk across four categories** — data, model, implementation,
   and regulatory — with regulatory risk at 45/100 being the lowest and most
   consequential.
3. **Produced a weighted compliance scorecard** with an overall score of 67.7
   and a rating of MODERATE — driven down by a fairness score of 40.
4. **Consolidated regulatory gaps** into a prioritized table with owners,
   timelines, and success criteria.
5. **Generated a compliance report** suitable for a Model Risk Committee, with
   a decision log covering fourteen project-level decisions.

**What it has not done:** it has not resolved any of the gaps. The fairness
finding remains open; IFRS 9 alignment is still pending; the foreign-data gap is
unchanged. That is correct — the governance report documents what needs to
happen; it does not do the remediation.

**The connection to the remediation story:** this notebook is the *document*
stage of the five-stage loop. It is where the findings from Notebooks 01 through
03 become a governance artifact with regulatory framing, priorities, and a
plan. What remains is the *monitor* stage — the ongoing process outside the
notebooks that tracks whether the mitigations are executed and whether the
model's performance holds up over time.

**The most important thing this notebook demonstrates:** a governance framework
that is willing to conclude "the model is conditional at best, and blocked from
deployment on fairness grounds." A compliance report that only ever certifies is
not a compliance report. This one identifies the gap that matters, attaches a
success criterion to it, and refuses to paper over it — which is exactly what
regulatory frameworks like OSFI E-23 and SR 11-7 are designed to require.